In [1]:
import pandas as pd


train_data = pd.read_csv("idea_novelty_checker/data/benchmark_data/train.csv")
test_data =  pd.read_csv("idea_novelty_checker/data/benchmark_data/test.csv")

train_data

,idea,domain,paper0_abstract,paper0_title,paper0_url,paper1_abstract,paper1_title,paper1_url,paper2_abstract,paper2_title,...,paper7_abstract,paper7_title,paper7_url,paper8_abstract,paper8_title,paper8_url,paper9_abstract,paper9_title,paper9_url,class
0,Develop a **joint analysis framework** designe...,"['Computer Science', 'Natural Language Process...",Peer review constitutes a core component of sc...,NLPeer: A Unified Resource for the Computation...,https://www.semanticscholar.org/paper/54a316ec...,"To assist human review process, we build a nov...",ReviewRobot: Explainable Paper Review Generati...,https://www.semanticscholar.org/paper/e7e46bc5...,The peer-review process is currently under str...,ReviVal: Towards Automatically Evaluating the ...,...,Expert feedback lays the foundation of rigorou...,Can large language models provide useful feedb...,https://www.semanticscholar.org/paper/f2209eb5...,"In this pilot study, we investigate the use of...",GPT4 is Slightly Helpful for Peer-Review Assis...,https://www.semanticscholar.org/paper/c17ca50b...,"Peer review is under pressure. Without fair, t...",Prophy: An automated reviewer finder to improv...,https://www.semanticscholar.org/paper/617a078b...,novel
1,Develop a **human-centric explainable AI syste...,"['Computer Science', 'Human-Computer Interacti...",Due to the cumbersome nature of human evaluati...,Who Validates the Validators? Aligning LLM-Ass...,https://www.semanticscholar.org/paper/6098ac10...,The SLAM paper demonstrated that on-device Sma...,Aligning Model Evaluations with Human Preferen...,https://www.semanticscholar.org/paper/b288562f...,Explainable artificially intelligent (XAI) sys...,Proxy tasks and subjective measures can be mis...,...,"Traditional reference-based metrics, such as B...",Human-Centered Design Recommendations for LLM-...,https://www.semanticscholar.org/paper/52570796...,From grading papers to summarizing medical doc...,ALLURE: A Systematic Protocol for Auditing and...,https://www.semanticscholar.org/paper/860bafe8...,"By simply composing prompts, developers can pr...",EvalLM: Interactive Evaluation of Large Langua...,https://www.semanticscholar.org/paper/a0d83f9e...,novel
2,Create an advanced literature search tool that...,"['Computer Science', 'Information Retrieval', ...",Citations allow quickly identifying related re...,PUREsuggest: Citation-based Literature Search ...,https://www.semanticscholar.org/paper/813fccf0...,PubMed is the most extensively used database a...,Efficacy improvement in searching MEDLINE data...,https://www.semanticscholar.org/paper/9a2d5eef...,Bibliographic data such as collections of scie...,Visual Analysis and Dissemination of Scientifi...,...,In order to help scholars understand and follo...,ComLittee: Literature Discovery with Personal ...,https://www.semanticscholar.org/paper/7f95d982...,There has been a proliferation of new research...,A typology of research discovery tools,https://www.semanticscholar.org/paper/760745c9...,Scientific silos can hinder innovation. These ...,Bridger: Toward Bursting Scientific Filter Bub...,https://www.semanticscholar.org/paper/4d02f1a1...,not novel
3,Implement a literature review tool that uses *...,"['Computer Science', 'Human-Computer Interacti...","When reading a scholarly article, inline citat...",CiteSee: Augmenting Citations in Scientific Pa...,https://www.semanticscholar.org/paper/81f7eb73...,A person often uses a single search engine for...,CiteSight: supporting contextual citation reco...,https://www.semanticscholar.org/paper/f6bea5e2...,Citations allow quickly identifying related re...,PUREsuggest: Citation-based Literature Search ...,...,Scholars who want to research a scientific top...,Relatedly: Scaffolding Literature Reviews with...,https://www.semanticscholar.org/paper/c388626d...,A literature review is a critical task in perf...,Papers101: Supporting the Discovery Process in...,https://www.semanticscholar.org/paper/adc5ab3d...,"When starting a new research activity, it 

In [2]:
train_data.shape

(35, 33)

In [3]:
novel = train_data[train_data['class'] == 'novel']
novel.shape

(20, 33)

In [4]:
notnovel = train_data[train_data['class'] == 'not novel']
notnovel.shape

(15, 33)

In [5]:
test_data.shape

(32, 33)

# Baseline Scores

This section evaluates three different prompting strategies:
1. **Prior Score**: Only the idea is provided to the LLM
2. **Posterior Zero-shot**: Idea + related papers
3. **Posterior Few-shot**: Idea + related papers + training examples

In [6]:
# Import required libraries
import os
import json
import time
from typing import List, Dict
import vertexai
from vertexai.generative_models import GenerativeModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

In [7]:
# Setup Vertex AI with service account
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'vertex_service_account.json'

# Read project ID from service account file
with open('vertex_service_account.json', 'r') as f:
    service_account_info = json.load(f)
    project_id = service_account_info['project_id']

# Initialize Vertex AI
vertexai.init(project=project_id, location='us-central1')

# Initialize the model
model = GenerativeModel('gemini-2.5-pro')

print(f"Initialized Vertex AI with project: {project_id}")

Initialized Vertex AI with project: nth-celerity-491206-g8


In [8]:
def get_related_papers_text(row, num_papers=10):
    """Extract related papers from a row."""
    papers_text = ""
    for i in range(num_papers):
        title = row.get(f'paper{i}_title', '')
        abstract = row.get(f'paper{i}_abstract', '')
        if pd.notna(title) and pd.notna(abstract):
            papers_text += f"\nPaper {i+1}:\nTitle: {title}\nAbstract: {abstract}\n"
    return papers_text

def parse_llm_response(response_text):
    """Parse LLM response to extract novel/not novel classification."""
    response_lower = response_text.lower().strip()
    
    # Check for explicit classifications
    if 'not novel' in response_lower:
        return 'not novel'
    elif 'novel' in response_lower:
        return 'novel'
    else:
        # Default to novel if unclear
        return 'novel'

def query_gemini(prompt, max_retries=3):
    """Query Gemini with retry logic."""
    for attempt in range(max_retries):
        try:
            response = model.generate_content(prompt)
            return response.text
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
                continue
            else:
                print(f"Error after {max_retries} attempts: {e}")
                return "novel"  # Default fallback

def calculate_metrics(y_true, y_pred):
    """Calculate accuracy, precision, recall, and F1 score."""
    # Convert to binary (1 for novel, 0 for not novel)
    y_true_binary = [1 if y == 'novel' else 0 for y in y_true]
    y_pred_binary = [1 if y == 'novel' else 0 for y in y_pred]
    
    accuracy = accuracy_score(y_true_binary, y_pred_binary)
    precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
    recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print("Helper functions defined.")

Helper functions defined.


## 1. Prior Score (Idea Only)

This approach only provides the idea to the LLM and asks whether it's novel or not, without any context about related papers.

In [9]:
# Prior Score: Idea only
print("Running Prior Score evaluation (Idea only)...")
print(f"Test set size: {len(test_data)}")

prior_predictions = []
y_true = test_data['class'].tolist()

for idx, row in tqdm(test_data.iterrows(), total=len(test_data), desc="Prior Score"):
    idea = row['idea']
    
    prompt = f"""You are an expert research evaluator. Your task is to determine if the following research idea is novel or not novel.

Research Idea:
{idea}

Please analyze this idea and respond with ONLY one of these two classifications:
- "novel" if the idea presents new, original concepts or approaches
- "not novel" if the idea is incremental, derivative, or already well-explored

Provide your classification:"""
    
    response = query_gemini(prompt)
    prediction = parse_llm_response(response)
    prior_predictions.append(prediction)
    
    # Rate limiting
    time.sleep(1)

# Calculate metrics
prior_metrics = calculate_metrics(y_true, prior_predictions)

print("\n" + "="*50)
print("PRIOR SCORE RESULTS (Idea Only)")
print("="*50)
print(f"Accuracy:  {prior_metrics['accuracy']:.4f}")
print(f"Precision: {prior_metrics['precision']:.4f}")
print(f"Recall:    {prior_metrics['recall']:.4f}")
print(f"F1 Score:  {prior_metrics['f1']:.4f}")
print("="*50)

Running Prior Score evaluation (Idea only)...
Test set size: 32


Prior Score: 100%|██████████| 32/32 [08:13<00:00, 15.42s/it]


PRIOR SCORE RESULTS (Idea Only)
Accuracy:  0.5312
Precision: 0.6429
Recall:    0.4737
F1 Score:  0.5455


## 2. Posterior Score - Zero-shot (Idea + Related Papers)

This approach provides both the idea and related papers to the LLM, allowing it to make a more informed decision.

In [10]:
# Posterior Zero-shot Score: Idea + Related Papers
print("Running Posterior Zero-shot evaluation (Idea + Related Papers)...")
print(f"Test set size: {len(test_data)}")

posterior_zero_predictions = []

for idx, row in tqdm(test_data.iterrows(), total=len(test_data), desc="Posterior Zero-shot"):
    idea = row['idea']
    papers = get_related_papers_text(row)
    
    prompt = f"""You are an expert research evaluator. Your task is to determine if a research idea is novel or not novel, given the idea and related papers in the field.

Research Idea:
{idea}

Related Papers:
{papers}

Instructions:
1. Carefully read the research idea
2. Review the related papers to understand the current state of research
3. Determine if the idea presents something truly new compared to existing work

Please respond with ONLY one of these two classifications:
- "novel" if the idea presents new, original concepts not covered by the related papers
- "not novel" if the idea is already addressed, similar to, or a minor variation of existing work

Provide your classification:"""
    
    response = query_gemini(prompt)
    prediction = parse_llm_response(response)
    posterior_zero_predictions.append(prediction)
    
    # Rate limiting
    time.sleep(1)

# Calculate metrics
posterior_zero_metrics = calculate_metrics(y_true, posterior_zero_predictions)

print("\n" + "="*50)
print("POSTERIOR ZERO-SHOT RESULTS (Idea + Related Papers)")
print("="*50)
print(f"Accuracy:  {posterior_zero_metrics['accuracy']:.4f}")
print(f"Precision: {posterior_zero_metrics['precision']:.4f}")
print(f"Recall:    {posterior_zero_metrics['recall']:.4f}")
print(f"F1 Score:  {posterior_zero_metrics['f1']:.4f}")
print("="*50)

Running Posterior Zero-shot evaluation (Idea + Related Papers)...
Test set size: 32


Posterior Zero-shot: 100%|██████████| 32/32 [10:03<00:00, 18.85s/it]


POSTERIOR ZERO-SHOT RESULTS (Idea + Related Papers)
Accuracy:  0.7188
Precision: 0.8125
Recall:    0.6842
F1 Score:  0.7429


## 3. Posterior Score - Few-shot (Idea + Related Papers + Training Examples)

This approach provides the idea, related papers, AND examples from the training set to help the LLM learn the classification pattern.

In [11]:
# Posterior Few-shot Score: Idea + Related Papers + Training Examples
print("Running Posterior Few-shot evaluation (Idea + Papers + Training Examples)...")
print(f"Test set size: {len(test_data)}")

# Create few-shot examples from training data (sample for efficiency)
# Use balanced samples: some novel, some not novel
num_examples = 5  # Number of examples per class
novel_examples = train_data[train_data['class'] == 'novel'].sample(n=min(num_examples, len(train_data[train_data['class'] == 'novel'])), random_state=42)
not_novel_examples = train_data[train_data['class'] == 'not novel'].sample(n=min(num_examples, len(train_data[train_data['class'] == 'not novel'])), random_state=42)
few_shot_examples = pd.concat([novel_examples, not_novel_examples])

# Build few-shot prompt prefix
few_shot_prefix = "Here are some examples of research ideas and their classifications:\n\n"

for idx, example_row in few_shot_examples.iterrows():
    example_idea = example_row['idea']
    example_papers = get_related_papers_text(example_row)
    example_class = example_row['class']
    
    few_shot_prefix += f"""Example:
Idea: {example_idea}

Related Papers:
{example_papers}

Classification: {example_class}

---

"""

posterior_few_predictions = []

for idx, row in tqdm(test_data.iterrows(), total=len(test_data), desc="Posterior Few-shot"):
    idea = row['idea']
    papers = get_related_papers_text(row)
    
    prompt = f"""You are an expert research evaluator. Your task is to determine if a research idea is novel or not novel.

{few_shot_prefix}

Now, classify the following research idea:

Research Idea:
{idea}

Related Papers:
{papers}

Based on the examples above and the related papers, please respond with ONLY one of these two classifications:
- "novel" if the idea presents new, original concepts
- "not novel" if the idea is already addressed or similar to existing work

Provide your classification:"""
    
    response = query_gemini(prompt)
    prediction = parse_llm_response(response)
    posterior_few_predictions.append(prediction)
    
    # Rate limiting
    time.sleep(1)

# Calculate metrics
posterior_few_metrics = calculate_metrics(y_true, posterior_few_predictions)

print("\n" + "="*50)
print("POSTERIOR FEW-SHOT RESULTS (Idea + Papers + Examples)")
print("="*50)
print(f"Accuracy:  {posterior_few_metrics['accuracy']:.4f}")
print(f"Precision: {posterior_few_metrics['precision']:.4f}")
print(f"Recall:    {posterior_few_metrics['recall']:.4f}")
print(f"F1 Score:  {posterior_few_metrics['f1']:.4f}")
print("="*50)

Running Posterior Few-shot evaluation (Idea + Papers + Training Examples)...
Test set size: 32


Posterior Few-shot: 100%|██████████| 32/32 [09:15<00:00, 17.35s/it]


POSTERIOR FEW-SHOT RESULTS (Idea + Papers + Examples)
Accuracy:  0.6250
Precision: 1.0000
Recall:    0.3684
F1 Score:  0.5385


## Summary: Comparison of All Three Approaches

In [12]:
# Compare all three approaches
import pandas as pd

results_df = pd.DataFrame({
    'Approach': ['Prior (Idea Only)', 'Posterior Zero-shot (Idea + Papers)', 'Posterior Few-shot (Idea + Papers + Examples)'],
    'Accuracy': [prior_metrics['accuracy'], posterior_zero_metrics['accuracy'], posterior_few_metrics['accuracy']],
    'Precision': [prior_metrics['precision'], posterior_zero_metrics['precision'], posterior_few_metrics['precision']],
    'Recall': [prior_metrics['recall'], posterior_zero_metrics['recall'], posterior_few_metrics['recall']],
    'F1 Score': [prior_metrics['f1'], posterior_zero_metrics['f1'], posterior_few_metrics['f1']]
})

print("\n" + "="*80)
print("BASELINE EVALUATION SUMMARY - GEMINI 2.5 PRO")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

# Save results
results_df.to_csv('baseline_results_gemini_2.5_pro.csv', index=False)
print("\nResults saved to: baseline_results_gemini_2.5_pro.csv")


BASELINE EVALUATION SUMMARY - GEMINI 2.5 PRO
                                     Approach  Accuracy  Precision   Recall  F1 Score
                            Prior (Idea Only)   0.53125   0.642857 0.473684  0.545455
          Posterior Zero-shot (Idea + Papers)   0.71875   0.812500 0.684211  0.742857
Posterior Few-shot (Idea + Papers + Examples)   0.62500   1.000000 0.368421  0.538462

Results saved to: baseline_results_gemini_2.5_pro.csv


In [13]:
# Save predictions for further analysis
predictions_df = pd.DataFrame({
    'true_label': y_true,
    'prior_prediction': prior_predictions,
    'posterior_zero_prediction': posterior_zero_predictions,
    'posterior_few_prediction': posterior_few_predictions
})

predictions_df.to_csv('baseline_predictions_gemini_2.5_pro.csv', index=False)
print("Predictions saved to: baseline_predictions_gemini_2.5_pro.csv")
predictions_df.head(10)

Predictions saved to: baseline_predictions_gemini_2.5_pro.csv


,true_label,prior_prediction,posterior_zero_prediction,posterior_few_prediction
0,not novel,novel,not novel,not novel
1,novel,novel,not novel,not novel
2,not novel,novel,not novel,not novel
3,not novel,not novel,novel,not novel
4,not novel,not novel,not novel,not novel
5,novel,not novel,not novel,not novel
6,not novel,not novel,not novel,not novel
7,novel,novel,novel,novel
8,not novel,novel,novel,not novel
9,not novel,novel,not novel,not novel


# New Experiments

This section evaluates two new prompting strategies:
1. **Few-shot with Reasoning**: Add reasoning/explanation to few-shot examples
2. **RAG-based Few-shot**: Use embeddings to retrieve k most similar examples (with and without reasoning)

## Experiment 1: Few-shot with Reasoning

This approach includes reasoning/explanation along with classification in few-shot examples to help the LLM understand the decision-making process.

In [14]:
# Few-shot with Reasoning
print("Running Few-shot with Reasoning evaluation...")
print(f"Test set size: {len(test_data)}")

# Create few-shot examples with reasoning
num_examples = 5
novel_examples = train_data[train_data['class'] == 'novel'].sample(n=min(num_examples, len(train_data[train_data['class'] == 'novel'])), random_state=42)
not_novel_examples = train_data[train_data['class'] == 'not novel'].sample(n=min(num_examples, len(train_data[train_data['class'] == 'not novel'])), random_state=42)
few_shot_examples_with_reasoning = pd.concat([novel_examples, not_novel_examples])

# Generate reasoning for each example using LLM
examples_with_reasoning = []
print("Generating reasoning for training examples...")
for idx, example_row in tqdm(few_shot_examples_with_reasoning.iterrows(), total=len(few_shot_examples_with_reasoning), desc="Generating reasoning"):
    example_idea = example_row['idea']
    example_papers = get_related_papers_text(example_row)
    example_class = example_row['class']
    
    # Generate reasoning
    reasoning_prompt = f"""You are an expert research evaluator. Given a research idea, related papers, and its classification, explain why this idea is classified as '{example_class}'.

Research Idea:
{example_idea}

Related Papers:
{example_papers}

Classification: {example_class}

Please provide a concise explanation (2-3 sentences) of why this idea is classified as '{example_class}'. Focus on:
- Whether the idea introduces new concepts not present in the related papers (for novel)
- Whether the idea is already addressed or very similar to existing work in the papers (for not novel)

Reasoning:"""
    
    reasoning = query_gemini(reasoning_prompt)
    examples_with_reasoning.append({
        'idea': example_idea,
        'papers': example_papers,
        'class': example_class,
        'reasoning': reasoning.strip()
    })
    time.sleep(1)

# Build few-shot prompt with reasoning
few_shot_reasoning_prefix = "Here are some examples of research ideas and their classifications with reasoning:\n\n"

for example in examples_with_reasoning:
    few_shot_reasoning_prefix += f"""Example:
Idea: {example['idea']}

Related Papers:
{example['papers']}

Classification: {example['class']}
Reasoning: {example['reasoning']}

---

"""

# Run evaluation
few_shot_reasoning_predictions = []

for idx, row in tqdm(test_data.iterrows(), total=len(test_data), desc="Few-shot with Reasoning"):
    idea = row['idea']
    papers = get_related_papers_text(row)
    
    prompt = f"""You are an expert research evaluator. Your task is to determine if a research idea is novel or not novel.

{few_shot_reasoning_prefix}

Now, classify the following research idea:

Research Idea:
{idea}

Related Papers:
{papers}

Based on the examples above and the related papers, please respond with ONLY one of these two classifications:
- "novel" if the idea presents new, original concepts
- "not novel" if the idea is already addressed or similar to existing work

You may optionally provide brief reasoning, but end with a clear classification.

Classification:"""
    
    response = query_gemini(prompt)
    prediction = parse_llm_response(response)
    few_shot_reasoning_predictions.append(prediction)
    time.sleep(1)


Running Few-shot with Reasoning evaluation...
Test set size: 32
Generating reasoning for training examples...


Generating reasoning:   0%|          | 0/10 [00:00<?, ?it/s]

Few-shot with Reasoning: 100%|██████████| 32/32 [09:34<00:00, 17.95s/it]


In [15]:
y_true = test_data['class'].tolist()
# Calculate metrics
few_shot_reasoning_metrics = calculate_metrics(y_true, few_shot_reasoning_predictions)

print("\n" + "="*50)
print("FEW-SHOT WITH REASONING RESULTS")
print("="*50)
print(f"Accuracy:  {few_shot_reasoning_metrics['accuracy']:.4f}")
print(f"Precision: {few_shot_reasoning_metrics['precision']:.4f}")
print(f"Recall:    {few_shot_reasoning_metrics['recall']:.4f}")
print(f"F1 Score:  {few_shot_reasoning_metrics['f1']:.4f}")
print("="*50)


FEW-SHOT WITH REASONING RESULTS
Accuracy:  0.7812
Precision: 0.8750
Recall:    0.7368
F1 Score:  0.8000


## Experiment 2: RAG-based Few-shot (without reasoning)

This approach uses embeddings to find the k most similar training examples to the test idea, then uses those examples for few-shot learning.

In [16]:
# Install and import required libraries for embeddings
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from vertexai.language_models import TextEmbeddingModel

# Initialize embedding model
embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-004")

def get_embedding(text):
    """Get embedding for a text using Vertex AI."""
    embeddings = embedding_model.get_embeddings([text])
    return embeddings[0].values

def find_similar_examples(query_idea, train_data, k=5):
    """Find k most similar training examples to the query idea using embeddings."""
    # Get embedding for query
    query_embedding = get_embedding(query_idea)
    
    # Compute similarities
    similarities = []
    for idx, row in train_data.iterrows():
        train_idea = row['idea']
        train_embedding = get_embedding(train_idea)
        similarity = cosine_similarity([query_embedding], [train_embedding])[0][0]
        similarities.append((idx, similarity))
    
    # Sort by similarity and get top k
    similarities.sort(key=lambda x: x[1], reverse=True)
    top_k_indices = [idx for idx, _ in similarities[:k]]
    
    return train_data.loc[top_k_indices]

print("RAG setup complete.")

RAG setup complete.


In [17]:
# RAG-based Few-shot (without reasoning)
print("Running RAG-based Few-shot evaluation (without reasoning)...")
print(f"Test set size: {len(test_data)}")

k = 5  # Number of similar examples to retrieve
rag_predictions = []

for idx, row in tqdm(test_data.iterrows(), total=len(test_data), desc="RAG Few-shot (no reasoning)"):
    idea = row['idea']
    papers = get_related_papers_text(row)
    
    # Find k most similar examples
    similar_examples = find_similar_examples(idea, train_data, k=k)
    
    # Build prompt with similar examples
    rag_prefix = f"Here are {k} similar research ideas and their classifications:\n\n"
    
    for _, example_row in similar_examples.iterrows():
        example_idea = example_row['idea']
        example_papers = get_related_papers_text(example_row)
        example_class = example_row['class']
        
        rag_prefix += f"""Example:
Idea: {example_idea}

Related Papers:
{example_papers}

Classification: {example_class}

---

"""
    
    prompt = f"""You are an expert research evaluator. Your task is to determine if a research idea is novel or not novel.

{rag_prefix}

Now, classify the following research idea:

Research Idea:
{idea}

Related Papers:
{papers}

Based on the similar examples above and the related papers, please respond with ONLY one of these two classifications:
- "novel" if the idea presents new, original concepts
- "not novel" if the idea is already addressed or similar to existing work

Provide your classification:"""
    
    response = query_gemini(prompt)
    prediction = parse_llm_response(response)
    rag_predictions.append(prediction)
    time.sleep(1)

# Calculate metrics
rag_metrics = calculate_metrics(y_true, rag_predictions)

print("\n" + "="*50)
print(f"RAG-BASED FEW-SHOT RESULTS (k={k}, no reasoning)")
print("="*50)
print(f"Accuracy:  {rag_metrics['accuracy']:.4f}")
print(f"Precision: {rag_metrics['precision']:.4f}")
print(f"Recall:    {rag_metrics['recall']:.4f}")
print(f"F1 Score:  {rag_metrics['f1']:.4f}")
print("="*50)

Running RAG-based Few-shot evaluation (without reasoning)...
Test set size: 32


RAG Few-shot (no reasoning): 100%|██████████| 32/32 [19:12<00:00, 36.02s/it]


RAG-BASED FEW-SHOT RESULTS (k=5, no reasoning)
Accuracy:  0.7188
Precision: 1.0000
Recall:    0.5263
F1 Score:  0.6897


## Experiment 3: RAG-based Few-shot with Reasoning

This approach combines RAG-based retrieval with reasoning in the few-shot examples.

In [18]:
# First, generate reasoning for all training examples (cache them)
print("Generating reasoning for all training examples...")
train_data_with_reasoning = []

for idx, example_row in tqdm(train_data.iterrows(), total=len(train_data), desc="Caching reasoning"):
    example_idea = example_row['idea']
    example_papers = get_related_papers_text(example_row)
    example_class = example_row['class']
    
    # Generate reasoning
    reasoning_prompt = f"""You are an expert research evaluator. Given a research idea, related papers, and its classification, explain why this idea is classified as '{example_class}'.

Research Idea:
{example_idea}

Related Papers:
{example_papers}

Classification: {example_class}

Please provide a concise explanation (2-3 sentences) of why this idea is classified as '{example_class}'. Focus on:
- Whether the idea introduces new concepts not present in the related papers (for novel)
- Whether the idea is already addressed or very similar to existing work in the papers (for not novel)

Reasoning:"""
    
    reasoning = query_gemini(reasoning_prompt)
    train_data_with_reasoning.append({
        'idx': idx,
        'idea': example_idea,
        'papers': example_papers,
        'class': example_class,
        'reasoning': reasoning.strip()
    })
    time.sleep(1)

print("Reasoning cached for all training examples.")

Generating reasoning for all training examples...


Caching reasoning: 100%|██████████| 35/35 [10:40<00:00, 18.31s/it]

Reasoning cached for all training examples.


In [19]:
# RAG-based Few-shot with Reasoning
print("Running RAG-based Few-shot evaluation (with reasoning)...")
print(f"Test set size: {len(test_data)}")

k = 5  # Number of similar examples to retrieve
rag_reasoning_predictions = []

for idx, row in tqdm(test_data.iterrows(), total=len(test_data), desc="RAG Few-shot (with reasoning)"):
    idea = row['idea']
    papers = get_related_papers_text(row)
    
    # Find k most similar examples
    similar_examples = find_similar_examples(idea, train_data, k=k)
    
    # Build prompt with similar examples and their reasoning
    rag_reasoning_prefix = f"Here are {k} similar research ideas and their classifications with reasoning:\n\n"
    
    for _, example_row in similar_examples.iterrows():
        # Find the cached reasoning for this example
        example_idx = example_row.name
        cached_example = next((e for e in train_data_with_reasoning if e['idx'] == example_idx), None)
        
        if cached_example:
            rag_reasoning_prefix += f"""Example:
Idea: {cached_example['idea']}

Related Papers:
{cached_example['papers']}

Classification: {cached_example['class']}
Reasoning: {cached_example['reasoning']}

---

"""
    
    prompt = f"""You are an expert research evaluator. Your task is to determine if a research idea is novel or not novel.

{rag_reasoning_prefix}

Now, classify the following research idea:

Research Idea:
{idea}

Related Papers:
{papers}

Based on the similar examples above and the related papers, please respond with ONLY one of these two classifications:
- "novel" if the idea presents new, original concepts
- "not novel" if the idea is already addressed or similar to existing work

You may optionally provide brief reasoning, but end with a clear classification.

Classification:"""
    
    response = query_gemini(prompt)
    prediction = parse_llm_response(response)
    rag_reasoning_predictions.append(prediction)
    time.sleep(1)

# Calculate metrics
rag_reasoning_metrics = calculate_metrics(y_true, rag_reasoning_predictions)

print("\n" + "="*50)
print(f"RAG-BASED FEW-SHOT WITH REASONING RESULTS (k={k})")
print("="*50)
print(f"Accuracy:  {rag_reasoning_metrics['accuracy']:.4f}")
print(f"Precision: {rag_reasoning_metrics['precision']:.4f}")
print(f"Recall:    {rag_reasoning_metrics['recall']:.4f}")
print(f"F1 Score:  {rag_reasoning_metrics['f1']:.4f}")
print("="*50)

Running RAG-based Few-shot evaluation (with reasoning)...
Test set size: 32


RAG Few-shot (with reasoning): 100%|██████████| 32/32 [20:58<00:00, 39.34s/it]


RAG-BASED FEW-SHOT WITH REASONING RESULTS (k=5)
Accuracy:  0.8750
Precision: 0.9412
Recall:    0.8421
F1 Score:  0.8889


## Comparison of All Experiments

In [20]:
# Compare all experiments including new ones
all_results_df = pd.DataFrame({
    'Approach': [
        'Prior (Idea Only)',
        'Posterior Zero-shot (Idea + Papers)',
        'Posterior Few-shot (Idea + Papers + Examples)',
        'Few-shot with Reasoning',
        f'RAG Few-shot (k={k}, no reasoning)',
        f'RAG Few-shot (k={k}, with reasoning)'
    ],
    'Accuracy': [
        prior_metrics['accuracy'],
        posterior_zero_metrics['accuracy'],
        posterior_few_metrics['accuracy'],
        few_shot_reasoning_metrics['accuracy'],
        rag_metrics['accuracy'],
        rag_reasoning_metrics['accuracy']
    ],
    'Precision': [
        prior_metrics['precision'],
        posterior_zero_metrics['precision'],
        posterior_few_metrics['precision'],
        few_shot_reasoning_metrics['precision'],
        rag_metrics['precision'],
        rag_reasoning_metrics['precision']
    ],
    'Recall': [
        prior_metrics['recall'],
        posterior_zero_metrics['recall'],
        posterior_few_metrics['recall'],
        few_shot_reasoning_metrics['recall'],
        rag_metrics['recall'],
        rag_reasoning_metrics['recall']
    ],
    'F1 Score': [
        prior_metrics['f1'],
        posterior_zero_metrics['f1'],
        posterior_few_metrics['f1'],
        few_shot_reasoning_metrics['f1'],
        rag_metrics['f1'],
        rag_reasoning_metrics['f1']
    ]
})

print("\n" + "="*80)
print("COMPLETE EVALUATION SUMMARY - ALL EXPERIMENTS")
print("="*80)
print(all_results_df.to_string(index=False))
print("="*80)

# Save results
all_results_df.to_csv('all_experiments_results_gemini_2.5_pro.csv', index=False)
print("\nResults saved to: all_experiments_results_gemini_2.5_pro.csv")


COMPLETE EVALUATION SUMMARY - ALL EXPERIMENTS
                                     Approach  Accuracy  Precision   Recall  F1 Score
                            Prior (Idea Only)   0.53125   0.642857 0.473684  0.545455
          Posterior Zero-shot (Idea + Papers)   0.71875   0.812500 0.684211  0.742857
Posterior Few-shot (Idea + Papers + Examples)   0.62500   1.000000 0.368421  0.538462
                      Few-shot with Reasoning   0.78125   0.875000 0.736842  0.800000
             RAG Few-shot (k=5, no reasoning)   0.71875   1.000000 0.526316  0.689655
           RAG Few-shot (k=5, with reasoning)   0.87500   0.941176 0.842105  0.888889

Results saved to: all_experiments_results_gemini_2.5_pro.csv


In [21]:
# Save all predictions for further analysis
all_predictions_df = pd.DataFrame({
    'true_label': y_true,
    'prior_prediction': prior_predictions,
    'posterior_zero_prediction': posterior_zero_predictions,
    'posterior_few_prediction': posterior_few_predictions,
    'few_shot_reasoning_prediction': few_shot_reasoning_predictions,
    'rag_prediction': rag_predictions,
    'rag_reasoning_prediction': rag_reasoning_predictions
})

all_predictions_df.to_csv('all_experiments_predictions_gemini_2.5_pro.csv', index=False)
print("All predictions saved to: all_experiments_predictions_gemini_2.5_pro.csv")
all_predictions_df.head(10)

All predictions saved to: all_experiments_predictions_gemini_2.5_pro.csv


,true_label,prior_prediction,posterior_zero_prediction,posterior_few_prediction,few_shot_reasoning_prediction,rag_prediction,rag_reasoning_prediction
0,not novel,novel,not novel,not novel,not novel,not novel,not novel
1,novel,novel,not novel,not novel,not novel,not novel,novel
2,not novel,novel,not novel,not novel,not novel,not novel,not novel
3,not novel,not novel,novel,not novel,novel,not novel,not novel
4,not novel,not novel,not novel,not novel,not novel,not novel,not novel
5,novel,not novel,not novel,not novel,not novel,novel,novel
6,not novel,not novel,not novel,not novel,not novel,not novel,not novel
7,novel,novel,novel,novel,novel,novel,novel
8,not novel,novel,novel,not novel,novel,not novel,not novel
9,not novel,novel,not novel,not novel,not novel,not novel,novel


## Author-Stated Novelty Claim Extraction (Train + Test)

This section extracts novelty claims for each paper appearing in train and test related-paper fields.

Extraction constraints:
- Claims must be explicitly stated by the paper author in the abstract text.
- Each claim must include a direct quote as evidence.
- A reason for the claim is extracted only when explicitly supported by text.
- If no explicit novelty claim exists, the model must return no claim.

In [22]:
# Novelty claim extraction utilities
import re
from typing import Any, Tuple


def collect_unique_papers(df: pd.DataFrame, max_papers_per_row: int = 10) -> pd.DataFrame:
    """Collect unique (title, abstract) papers from dataset rows."""
    records = []
    for row_idx, row in df.iterrows():
        for i in range(max_papers_per_row):
            title_col = f"paper{i}_title"
            abs_col = f"paper{i}_abstract"
            if title_col in df.columns and abs_col in df.columns:
                title = row.get(title_col, "")
                abstract = row.get(abs_col, "")
                if pd.notna(title) and pd.notna(abstract):
                    title = str(title).strip()
                    abstract = str(abstract).strip()
                    if title and abstract:
                        records.append({
                            "title": title,
                            "abstract": abstract,
                            "source_row": row_idx,
                            "paper_slot": i
                        })

    if not records:
        return pd.DataFrame(columns=["title", "abstract", "source_row", "paper_slot"])

    papers_df = pd.DataFrame(records)
    papers_df = papers_df.drop_duplicates(subset=["title", "abstract"]).reset_index(drop=True)
    return papers_df


def _extract_json_block(text: str) -> str:
    """Extract a JSON object from raw model output."""
    text = text.strip()
    fenced = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    if fenced:
        return fenced.group(1)
    plain = re.search(r"(\{.*\})", text, flags=re.DOTALL)
    if plain:
        return plain.group(1)
    return "{}"


def _contains_verbatim_quote(quote: str, source_text: str) -> bool:
    """Verify quote appears verbatim in source text (case-insensitive fallback)."""
    if not quote:
        return False
    q = quote.strip()
    s = source_text or ""
    return (q in s) or (q.lower() in s.lower())


def parse_claim_json(raw_response: str, abstract: str) -> Dict[str, Any]:
    """Parse and validate claim extraction response."""
    default = {
        "has_explicit_novelty_claim": False,
        "claim_text": "",
        "claim_quote": "",
        "reason_text": "",
        "reason_quote": "",
        "confidence": "low",
        "validation_note": "No valid explicit claim found."
    }

    try:
        payload = json.loads(_extract_json_block(raw_response))
    except Exception:
        return default

    has_claim = bool(payload.get("has_explicit_novelty_claim", False))
    claim_text = str(payload.get("claim_text", "")).strip()
    claim_quote = str(payload.get("claim_quote", "")).strip()
    reason_text = str(payload.get("reason_text", "")).strip()
    reason_quote = str(payload.get("reason_quote", "")).strip()
    confidence = str(payload.get("confidence", "low")).strip().lower()

    if confidence not in {"high", "medium", "low"}:
        confidence = "low"

    if not has_claim:
        return {
            **default,
            "validation_note": str(payload.get("validation_note", "No explicit novelty claim identified.")).strip()
        }

    # Enforce factuality: evidence quote must be present in abstract.
    if not _contains_verbatim_quote(claim_quote, abstract):
        return {
            **default,
            "validation_note": "Claim quote was not found verbatim in abstract; dropped to avoid hallucination."
        }

    # Reason quote is optional but must be supported if present.
    if reason_quote and not _contains_verbatim_quote(reason_quote, abstract):
        reason_text = ""
        reason_quote = ""

    return {
        "has_explicit_novelty_claim": True,
        "claim_text": claim_text,
        "claim_quote": claim_quote,
        "reason_text": reason_text,
        "reason_quote": reason_quote,
        "confidence": confidence,
        "validation_note": str(payload.get("validation_note", "")).strip()
    }


def extract_novelty_claim_for_paper(title: str, abstract: str) -> Dict[str, Any]:
    """Extract author-stated novelty claim and reason with strict evidence grounding."""
    prompt = f"""You are an evidence-grounded scientific information extraction assistant.

Task:
Extract novelty claims ONLY if they are explicitly stated by the paper author in the abstract.
Do NOT infer, speculate, or paraphrase beyond what is explicitly stated.

Return STRICT JSON with this schema:
{{
  "has_explicit_novelty_claim": true/false,
  "claim_text": "understandable one-sentence summary of the explicit author claim",
  "claim_quote": "exact verbatim quote from abstract supporting claim_text",
  "reason_text": "understandable one-sentence reason why authors say it is novel",
  "reason_quote": "exact verbatim quote from abstract supporting reason_text",
  "confidence": "high|medium|low",
  "validation_note": "brief note if claim is absent or uncertain"
}}

Rules:
1) If no explicit novelty statement exists, set has_explicit_novelty_claim=false and leave claim/reason fields empty.
2) claim_quote and reason_quote must be copied exactly from the abstract.
3) reason fields can be empty if no explicit reason is stated.
4) Keep claim_text and reason_text clear and understandable, but faithful to the quotes.
5) Output JSON only.

Paper Title:
{title}

Abstract:
{abstract}
"""

    raw = query_gemini(prompt)
    parsed = parse_claim_json(raw, abstract)
    parsed["raw_response"] = raw
    return parsed


# Build paper pools from train and test
train_papers = collect_unique_papers(train_data)
test_papers = collect_unique_papers(test_data)

print(f"Unique train papers: {len(train_papers)}")
print(f"Unique test papers: {len(test_papers)}")

Unique train papers: 264
Unique test papers: 231


In [25]:
# Run novelty claim extraction for train and test paper pools

def run_claim_extraction(papers_df: pd.DataFrame, split_name: str, sleep_seconds: float = 0.01) -> pd.DataFrame:
    outputs = []

    for idx, row in tqdm(papers_df.iterrows(), total=len(papers_df), desc=f"Extracting claims ({split_name})"):
        title = row["title"]
        abstract = row["abstract"]

        try:
            extraction = extract_novelty_claim_for_paper(title, abstract)
        except Exception as e:
            extraction = {
                "has_explicit_novelty_claim": False,
                "claim_text": "",
                "claim_quote": "",
                "reason_text": "",
                "reason_quote": "",
                "confidence": "low",
                "validation_note": f"Extraction error: {e}",
                "raw_response": ""
            }

        outputs.append({
            "split": split_name,
            "paper_index": idx,
            "title": title,
            "abstract": abstract,
            **extraction
        })

        time.sleep(sleep_seconds)

    return pd.DataFrame(outputs)


train_claims_df = run_claim_extraction(train_papers, split_name="train")
test_claims_df = run_claim_extraction(test_papers, split_name="test")
all_claims_df = pd.concat([train_claims_df, test_claims_df], ignore_index=True)

# Save extraction outputs
all_claims_df.to_csv("paper_novelty_claims_train_test_gemini_2.5_pro.csv", index=False)
print("Saved: paper_novelty_claims_train_test_gemini_2.5_pro.csv")

# Save a cleaner, evidence-focused view
clean_cols = [
    "split", "paper_index", "title",
    "has_explicit_novelty_claim", "claim_text", "claim_quote",
    "reason_text", "reason_quote", "confidence", "validation_note"
]
all_claims_df[clean_cols].to_csv("paper_novelty_claims_evidence_view.csv", index=False)
print("Saved: paper_novelty_claims_evidence_view.csv")

# Quick quality summary
summary = {
    "total_papers": len(all_claims_df),
    "with_explicit_claim": int(all_claims_df["has_explicit_novelty_claim"].sum()),
    "without_explicit_claim": int((~all_claims_df["has_explicit_novelty_claim"]).sum()),
}
print(summary)

all_claims_df[clean_cols].head(10)

Extracting claims (test):  96%|█████████▌| 222/231 [1:21:28<03:18, 22.02s/it]


KeyboardInterrupt: 

In [26]:
all_claims_df = train_claims_df
# Save extraction outputs
all_claims_df.to_csv("paper_novelty_claims_train_test_gemini_2.5_pro.csv", index=False)
print("Saved: paper_novelty_claims_train_test_gemini_2.5_pro.csv")

# Save a cleaner, evidence-focused view
clean_cols = [
    "split", "paper_index", "title",
    "has_explicit_novelty_claim", "claim_text", "claim_quote",
    "reason_text", "reason_quote", "confidence", "validation_note"
]
all_claims_df[clean_cols].to_csv("paper_novelty_claims_evidence_view.csv", index=False)
print("Saved: paper_novelty_claims_evidence_view.csv")

# Quick quality summary
summary = {
    "total_papers": len(all_claims_df),
    "with_explicit_claim": int(all_claims_df["has_explicit_novelty_claim"].sum()),
    "without_explicit_claim": int((~all_claims_df["has_explicit_novelty_claim"]).sum()),
}
print(summary)

all_claims_df[clean_cols].head(10)

Saved: paper_novelty_claims_train_test_gemini_2.5_pro.csv
Saved: paper_novelty_claims_evidence_view.csv
{'total_papers': 264, 'with_explicit_claim': 159, 'without_explicit_claim': 105}


,split,paper_index,title,has_explicit_novelty_claim,claim_text,claim_quote,reason_text,reason_quote,confidence,validation_note
0,train,0,NLPeer: A Unified Resource for the Computation...,True,"The authors introduce NLPeer, the first ethica...","To remedy this, we introduce NLPeer– the first...",This resource is novel because the lack of cle...,the lack of clearly licensed datasets and mult...,high,The abstract also mentions a 'novel guided ski...
1,train,1,ReviewRobot: Explainable Paper Review Generati...,True,The authors have built a novel system called R...,"To assist human review process, we build a nov...",,,high,The abstract explicitly states that the 'Revie...
2,train,2,ReviVal: Towards Automatically Evaluating the ...,True,The authors claim their work is the first of i...,"To the best of our knowledge, our work is a fi...",The work is novel because it addresses a task ...,Our approach significantly outperforms several...,high,
3,train,3,Can We Automate Scientific Reviewing?,False,,,,,low,"The abstract describes the work performed, inc..."
4,train,4,A Dataset of Peer Reviews (PeerRead): Collecti...,True,The authors present the first publicly availab...,We present the first public dataset of scienti...,,,high,The claim is explicitly stated using the phras...
5,train,5,KID-Review: Knowledge-Guided Scientific Review...,True,This paper presents the first step in explorin...,"In this paper, as a first step, we explore the...",,,high,The abstract explicitly states 'as a first ste...
6,train,6,ReviewerGPT? An Exploratory Study on Using Lar...,False,,,,,low,The abstract describes an 'exploratory study' ...
7,train,7,Can large language models provide useful feedb...,True,This paper presents a systematic study on the ...,"To address this gap, we created an automated p...",This work is novel because the usefulness of L...,"However, the utility of LLM-generated feedback...",high,The authors explicitly frame their work as a n...
8,train,8,GPT4 is Slightly Helpful for Peer-Review Assis...,True,The authors claim their findings open new aven...,Our findings open new avenues for leveraging m...,,,high,The abstract makes an explicit claim about the...
9,train,9,Prophy: An automated reviewer finder to improv...,True,"The authors developed Prophy, a state-of-the-a...",At Prophy we have developed a state-of-the-art...,The methods are presented as novel because the...,These methods can ensure fair and independent ...,high,The abstract explicitly uses the term 'state-o...


In [27]:
# Optional: inspect only high-confidence factual claims with evidence quotes
high_conf_claims = all_claims_df[
    (all_claims_df["has_explicit_novelty_claim"] == True) &
    (all_claims_df["confidence"].str.lower() == "high")
][[
    "split", "title", "claim_text", "claim_quote", "reason_text", "reason_quote"
]].reset_index(drop=True)


print(f"High-confidence explicit claims: {len(high_conf_claims)}")
high_conf_claims.head(20)

High-confidence explicit claims: 154


,split,title,claim_text,claim_quote,reason_text,reason_quote
0,train,NLPeer: A Unified Resource for the Computation...,"The authors introduce NLPeer, the first ethica...","To remedy this, we introduce NLPeer– the first...",This resource is novel because the lack of cle...,the lack of clearly licensed datasets and mult...
1,train,ReviewRobot: Explainable Paper Review Generati...,The authors have built a novel system called R...,"To assist human review process, we build a nov...",,
2,train,ReviVal: Towards Automatically Evaluating the ...,The authors claim their work is the first of i...,"To the best of our knowledge, our work is a fi...",The work is novel because it addresses a task ...,Our approach significantly outperforms several...
3,train,A Dataset of Peer Reviews (PeerRead): Collecti...,The authors present the first publicly availab...,We present the first public dataset of scienti...,,
4,train,KID-Review: Knowledge-Guided Scientific Review...,This paper presents the first step in explorin...,"In this paper, as a first step, we explore the...",,
5,train,Can large language models provide useful feedb...,This paper presents a systematic study on the ...,"To address this gap, we created an automated p...",This work is novel because the usefulness of L...,"However, the utility of LLM-generated feedback..."
6,train,GPT4 is Slightly Helpful for Peer-Review Assis...,The authors claim their findings open new aven...,Our findings open new avenues for leveraging m...,,
7,train,Prophy: An automated reviewer finder to improv...,"The authors developed Prophy, a state-of-the-a...",At Prophy we have developed a state-of-the-art...,The methods are presented as novel because the...,These methods can ensure fair and independent ...
8,train,Who Validates the Validators? Aligning LLM-Ass...,The paper presents a mixed-initiative approach...,We present a mixed-initiative approach to ``va...,The authors also identify and name a new pheno...,"In particular, we identify a phenomenon we dub..."
9,train,Aligning Model Evaluations with Human Preferen...,The authors developed a recalibration procedur...,We employed Bayesian statistics and a t-test t...,This procedure was developed to align LLM eval...,This follow-up paper explores methods to align...
